In [3]:
import json
import os
import requests
import pandas as pd


# ============================================================
# 1. API 설정
# ============================================================

API_URL = (
    "https://api-stats.kbl.or.kr/api/records/player/"
    "general/traditional"
)

PARAMS = {
    "seasonCode": "47",
    "gameCode": "01",
    "sortDataSc": "SCORE",
    "sortOrderSc": "desc",

    # 한 번에 충분히 많이 받도록 설정
    "listCn": "100",
    "pageNo": "1",

    "ruleCk": "1",

    # teamCode=35는 삼성으로 보이지만
    # 응답 확인 후 확정
    "teamCode": "35",

    "perCn": "0",
    "lastCn": "0",
    "partIfList": "0",
    "draftNo": "0"
}

HEADERS = {
    "Accept": (
        "application/json, text/plain, */*"
    ),
    "Accept-Language": (
        "ko,en-US;q=0.9,en;q=0.8,"
        "zh-CN;q=0.7,zh;q=0.6"
    ),
    "Origin": (
        "https://www.kbl.or.kr"
    ),
    "Referer": (
        "https://www.kbl.or.kr/"
    ),
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    )
}


# ============================================================
# 2. API 요청
# ============================================================

response = requests.get(
    API_URL,
    params=PARAMS,
    headers=HEADERS,
    timeout=30
)

print("상태 코드:", response.status_code)
print("최종 URL:", response.url)
print(
    "Content-Type:",
    response.headers.get(
        "content-type",
        ""
    )
)
print("응답 길이:", len(response.text))

response.raise_for_status()


# ============================================================
# 3. JSON 응답 확인
# ============================================================

try:
    data = response.json()

except ValueError:
    print(response.text[:2000])

    raise ValueError(
        "JSON 응답이 아닙니다. "
        "API 요청 또는 헤더를 확인하세요."
    )


print("\n===== JSON 최상위 구조 =====")

if isinstance(data, dict):
    print(data.keys())

elif isinstance(data, list):
    print(
        "응답 형식: list, 행 수:",
        len(data)
    )

else:
    print(
        "응답 형식:",
        type(data)
    )


print("\n===== JSON 응답 일부 =====")
print(
    json.dumps(
        data,
        ensure_ascii=False,
        indent=2
    )[:3000]
)


# ============================================================
# 4. 선수 기록 목록 자동 탐색
# ============================================================

def find_list_of_dicts(obj):
    """
    JSON 안에서 선수 기록으로 보이는
    dict 리스트를 재귀적으로 탐색
    """

    if isinstance(obj, list):

        if len(obj) == 0:
            return obj

        if all(
            isinstance(item, dict)
            for item in obj
        ):
            return obj

        for item in obj:
            found = find_list_of_dicts(item)

            if found is not None:
                return found

    elif isinstance(obj, dict):

        priority_keys = [
            "data",
            "result",
            "list",
            "rows",
            "content",
            "playerList",
            "playerRecords",
            "recordList",
            "records",
            "items"
        ]

        for key in priority_keys:
            if key in obj:
                found = find_list_of_dicts(
                    obj[key]
                )

                if found is not None:
                    return found

        for value in obj.values():
            found = find_list_of_dicts(value)

            if found is not None:
                return found

    return None


records = find_list_of_dicts(data)

if records is None:
    raise ValueError(
        "JSON 안에서 선수 기록 리스트를 찾지 못했습니다."
    )

print(
    "\n찾은 선수 기록 행 수:",
    len(records)
)


# ============================================================
# 5. 중첩 JSON 평탄화
# ============================================================

def flatten_dict(
    obj,
    parent_key="",
    separator="_"
):
    result = {}

    if not isinstance(obj, dict):
        return {
            parent_key: obj
        }

    for key, value in obj.items():

        new_key = (
            f"{parent_key}{separator}{key}"
            if parent_key
            else str(key)
        )

        if isinstance(value, dict):
            result.update(
                flatten_dict(
                    value,
                    new_key,
                    separator
                )
            )

        elif isinstance(value, list):
            result[new_key] = json.dumps(
                value,
                ensure_ascii=False
            )

        else:
            result[new_key] = value

    return result


rows = [
    flatten_dict(record)
    for record in records
]

df = pd.DataFrame(rows)


# ============================================================
# 6. 컬럼명 확인
# ============================================================

print("\n===== 수집된 컬럼 =====")

print(
    df.columns.tolist()
)

print("\n===== 원자료 앞부분 =====")

print(
    df.head(10).to_string(
        index=False
    )
)


# ============================================================
# 7. 데이터 타입 정리
# ============================================================

# 숫자로 변환 가능한 컬럼은 숫자형으로 변환
for col in df.columns:

    if col in [
        "playerName",
        "player_name",
        "pname",
        "teamName",
        "team_name",
        "tname"
    ]:
        continue

    converted = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # 일정 비율 이상 숫자로 변환되면 숫자형으로 채택
    if converted.notna().mean() >= 0.7:
        df[col] = converted


# ============================================================
# 8. 삼성 팀 응답 여부 확인
# ============================================================

team_columns = [
    col
    for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "team",
            "tname",
            "teamname",
            "team_nm",
            "teamname"
        ]
    )
]

print(
    "\n팀 관련 컬럼:",
    team_columns
)

if team_columns:

    print("\n팀 컬럼별 고유값:")

    for col in team_columns:
        values = (
            df[col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        print(
            col,
            ":",
            values[:30]
        )


# ============================================================
# 9. 저장
# ============================================================

OUTPUT_FILE = (
    "samsung_2025_26_player_records_raw.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\n저장 완료:",
    os.path.abspath(OUTPUT_FILE)
)

상태 코드: 200
최종 URL: https://api-stats.kbl.or.kr/api/records/player/general/traditional?seasonCode=47&gameCode=01&sortDataSc=SCORE&sortOrderSc=desc&listCn=100&pageNo=1&ruleCk=1&teamCode=35&perCn=0&lastCn=0&partIfList=0&draftNo=0
Content-Type: application/json; charset=utf-8
응답 길이: 7564

===== JSON 최상위 구조 =====
dict_keys(['resultCode', 'message', 'data'])

===== JSON 응답 일부 =====
{
  "resultCode": 200,
  "message": "요청이 성공적으로 처리되었습니다.",
  "data": [
    {
      "seasonCode": 47,
      "rankNo": 1,
      "playerNo": "291787",
      "gameCode": null,
      "gameNo": null,
      "quarterGu": null,
      "kname": "케렘 칸터",
      "ename": "Kerem Kanter",
      "teamCode": "35",
      "teamName1": "서울 삼성",
      "teamName4": "SAMSUNG",
      "gameCount": 54,
      "win": 16,
      "lose": 38,
      "playSec": 76281.0,
      "score": 934.0,
      "fg": 302.0,
      "fgA": 559.0,
      "fgRt": 54.02504472271914,
      "threep": 25.0,
      "threepA": 67.0,
      "threepRt": 37.3134328358209,
      "